<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# 01_04 Ensembles 


## 1. Bagging and Random Forest

### Step 1.1: Bagging with Decision Trees

**Bagging** (bootstrap aggregating) trains many trees on different bootstrap samples and averages their predictions. This reduces variance and usually improves generalization over a single tree.

**What to do**
1. Create a `BaggingClassifier` with a `DecisionTreeClassifier(random_state=42)`.
2. Fit on the Breast Cancer training set.
3. Print train/test accuracy.

Has the accuracy improved with respect to using a single tree?


### Step 1.2: Random Forest (bagging + random subspace)

A Random Forest adds feature randomness at each split (`max_features`) and often improving performance further.

**What to do**
1. Fit `RandomForestClassifier` on the Breast Cancer training set.
2. Print train/test accuracy; plot the decision boundary.
3. Access and print `.feature_importances_`.
4. Sort importances in a descending order and plot a horizontal bar chart with feature names. Are there changes with respect to a single tree?

### Step 1.3: Study Random Forest parameters

Random Forests have several important hyperparameters that performance and interpretability.  
We’ll sweep a few to see their impact.

**What to do**
1. Number of trees (`n_estimators`) train forests with `[10, 50, 100, 200, 500]`; plot train/test accuracy vs the number of trees.
2. Splitting criterion: compare `criterion="gini"` (default) vs `criterion="entropy"`; report test accuracy for each.
3. Tree depth (`max_depth`): compare shallow forests (`max_depth=3`) vs unrestricted depth (`None`); see how it affects overfitting.
4. Minimum samples per leaf (`min_samples_leaf`): try values `[1, 5, 10]`; note how increasing this regularizes the forest.

In general, more trees stabilize performance, deeper trees risk overfitting, larger leaves and entropy/gini choices influence splits but often with minor differences.


## 2. Boosting

### Step 2.1: AdaBoost vs Gradient Boosting

Boosting builds trees sequentially: each new tree focuses on errors of the current ensemble. AdaBoost: reweights misclassified points; often uses shallow trees (decision stumps) and Gradient Boosting fits each tree to residuals. 

**What to do**
1. Fit an `AdaBoostClassifier` with a decision stump (tree with maximum depth of 1) as base classifier and a `GradientBoostingClassifier` on the Breast Cancer training set.
2. Report training and test accuracy for both.
3. Compare to Random Forest: which generalizes better on this dataset?


### Step 2.2: Visualizing AdaBoost base classifiers on moons

On 2D moons dataset, boosting is easy to visualize. We can actually see the sequential refinement.

**What to do**
1. Fit AdaBoost with stumps (`max_depth=1`) on the moons dataset with 5 base classifiers.  
2. Access `.estimators_`: the list of fitted base classifiers (here: 5 stumps).  
3. Plot the decision boundary for each stump separately.  
4. Finally, plot the decision boundary of the whole AdaBoost ensemble  

Each stump alone is weak, but together they capture the complex moons shape.  


### Step 2.3: XGBoost

[XGBoost](https://xgboost.readthedocs.io/en/stable/) (Extreme Gradient Boosting) is a highly optimized implementation of gradient-boosted trees.  
It is widely used in machine learning competitions (e.g. Kaggle) and real-world applications because it is fast, regularized, and accurate.

Two ways to use XGBoost
- Native API (`xgboost.train`): lower-level, more control.  
- Scikit-learn wrapper (`XGBClassifier`, `XGBRegressor`): works like any sklearn model, with `.fit`, `.predict`, `.score`.  
  - This is what we’ll use here.

#### Why is XGBoost good?
- Multiple parameters to improve generalization (`reg_lambda`, `reg_alpha`, `subsample`, `colsample_bytree`, etc.).
- Highly efficient implementation (multi-threaded, memory optimized).  
- Built-in support for **early stopping**. Instead of fixing the number of trees, XGBoost can monitor performance on a **validation set**.  
    - You train with a large `n_estimators` (e.g. 2000).  
    - If performance does not improve after `early_stopping_rounds` iterations, training stops early.  
    - The best number of trees is saved in `.best_ntree_limit`.

This means: we don’t have to guess the number of trees, XGBoost finds it automatically.

**What to do**
1. Import `XGBClassifier` from `xgboost`.  
2. Fit on the Breast Cancer dataset with parameters (`n_estimators=300, learning_rate=0.1, max_depth=3, subsample=0.8, colsample_bytree=0.8`). Use a validation split and `early_stopping_rounds=20` to automatically select the best number of trees.  
4. Report train/test accuracy; compare to Gradient Boosting and Random Forest.  
5. Inspect `.feature_importances_` and compare to Random Forest.  


## 3. Multi-class strategies: OVA and OVO

### Step 3.1: Create a synthetic 4-class dataset (2D)

To study multiclass strategies, we’ll generate a simple 4-class dataset in 2D using `make_blobs`.

**Exact configuration**
- 4 clusters centered at the corners of a square: `(-2, -2), (2, -2), (2, 2), (-2, 2)`
- 200 points per class (total 800 samples)
- Cluster standard deviation = `0.9` 
- Random seed = `7`
- Then apply a small linear transformation to tilt/stretch the blobs, so they are not perfectly axis-aligned. This is done multiplying the generated data by a 2x2 matrix. 

**What to do**
1. Use `make_blobs` with the parameters above to generate `(X_raw, y)`.  
2. Multiply `X_raw` by a matrix `A`. You can set `A = np.array([[-1,1], [0, 0.5]])`  
3. Split into train/test (70/30).  
4. Scatter-plot the training points with colors by class to check the structure.  
5. Define `class_names = ["class 0", "class 1", "class 2", "class 3"]` for later use.  


### Step 3.2: OvA vs OvO

Let's compare these two approaches to handle multi-class classification problems.

**What to do**
1. Choose a base learner. Since decision trees can naturally handle multiclass classification, we’ll use Decision Stumps, `max_depth=1`) to make the binary subproblems clear.
2. Fit `OneVsRestClassifier(base)` and `OneVsOneClassifier(base)` on the 4-class training set.
3. Report train/test accuracy for OvR and OvO.
4. Compare intermediate classifiers:
   - OvA: for each class, evaluate the binary task on train/test; report the results in a dataframe, stating which is the fixed class.
   - OvO: for each pair (i, j), restrict to samples of those two classes and evaluate the pairwise binary classifier; report the results in a dataframe, stating which pair of classes are being compared. The comparison order is (0 vs 1, 0 vs 2, 0 vs 3, 1 vs 2, 1 vs 3, 2 vs 3):  combinations(range(K), 2). 
5. Briefly interpret: which class is hardest vs rest? which pairs are most confusable? Relate this to the geometry of your scatter plot.
